In [2]:
import json
import pandas as pd
from datasets import load_dataset

#load the filtered hotel data we saved in Phase 1
import os
os.getcwd()

c:\Users\Hamza\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'c:\\Users\\Hamza\\hotel-dialogue-system\\notebooks'

In [3]:
# Load the filtered hotel training data we saved in Phase 1
hotel_train = pd.read_json("../data/multiwoz/hotel_train.json", lines=True)
hotel_val = pd.read_json("../data/multiwoz/hotel_val.json", lines=True)

print(f"Train: {len(hotel_train)} dialogues")
print(f"Val: {len(hotel_val)} dialogues")
hotel_train.head()

Train: 3369 dialogues
Val: 418 dialogues


,dialogue_id,services,turns
0,PMUL4398.json,"[restaurant, hotel]","{'turn_id': ['0', '1', '2', '3', '4', '5', '6'..."
1,SNG1013.json,[hotel],"{'turn_id': ['0', '1', '2', '3', '4', '5', '6'..."
2,PMUL0121.json,"[restaurant, hotel]","{'turn_id': ['0', '1', '2', '3', '4', '5', '6'..."
3,PMUL3484.json,"[restaurant, taxi, hotel]","{'turn_id': ['0', '1', '2', '3', '4', '5', '6'..."
4,PMUL2979.json,"[taxi, attraction, hotel]","{'turn_id': ['0', '1', '2', '3', '4', '5', '6'..."


In [4]:
# Extract user turns with hotel intents
rows = []

for _, dialogue in hotel_train.iterrows():
    turns = dialogue["turns"]
    
    for i, (speaker, utterance, frames) in enumerate(zip(
        turns["speaker"], turns["utterance"], turns["frames"]
    )):
        # Only look at user turns (speaker == 0)
        if speaker != 0:
            continue
        
        # Check each frame for hotel intent
        for j, state in enumerate(frames["state"]):
            intent = state["active_intent"]
            service = frames["service"][j]
            
            if service == "hotel" and intent in ["find_hotel", "book_hotel"]:
                rows.append({
                    "utterance": utterance,
                    "intent": intent,
                    "dialogue_id": dialogue["dialogue_id"]
                })

df = pd.DataFrame(rows)
print(f"Total hotel turns: {len(df)}")
print(df["intent"].value_counts())
df.head()

Total hotel turns: 14868
intent
find_hotel    10465
book_hotel     4403
Name: count, dtype: int64


,utterance,intent,dialogue_id
0,i need a place to dine in the center thats exp...,find_hotel,PMUL4398.json
1,"Any sort of food would be fine, as long as it ...",find_hotel,PMUL4398.json
2,"Sounds good, could I get that phone number? Al...",find_hotel,PMUL4398.json
3,Yes. Can you book it for me?,find_hotel,PMUL4398.json
4,i want to book it for 2 people and 2 nights st...,book_hotel,PMUL4398.json


In [5]:
# Drop sentences that are too short or clearly not about hotels
df = df[df["utterance"].str.split().str.len() > 3]

# Shuffle the data
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"After cleaning: {len(df)} turns")
print(df["intent"].value_counts())
df.head()

After cleaning: 14723 turns
intent
find_hotel    10389
book_hotel     4334
Name: count, dtype: int64


,utterance,intent,dialogue_id
0,Is that a 2 star hotel?,find_hotel,MUL0767.json
1,Is it in the north? I need to be in the north ...,find_hotel,SSNG0284.json
2,"Great, and what is the address? Also, can I ju...",find_hotel,MUL2528.json
3,I want to to have a 4 star rating & be a guest...,find_hotel,PMUL3849.json
4,How about a different hotel in the same price ...,book_hotel,SSNG0349.json


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Split data into train and test
X_train, X_test, y_train, y_test = train_test_split(
    df["utterance"], df["intent"],
    test_size=0.2,
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Training samples: 11778
Test samples: 2945


In [7]:
# Step 1 — convert text to TF-IDF vectors
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Step 2 — train logistic regression classifier
classifier = LogisticRegression(class_weight="balanced", max_iter=1000)
classifier.fit(X_train_tfidf, y_train)

print("Model trained.")

Model trained.


In [8]:
# Evaluate the classifier
y_pred = classifier.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

  book_hotel       0.78      0.86      0.82       860
  find_hotel       0.94      0.90      0.92      2085

    accuracy                           0.89      2945
   macro avg       0.86      0.88      0.87      2945
weighted avg       0.89      0.89      0.89      2945



In [9]:
# Test on custom sentences
test_sentences = [
    "I need a cheap hotel in the centre",
    "Can you book it for 2 nights for 3 people",
    "I am looking for a guesthouse with free parking",
    "Please make a reservation for monday"
]

# Vectorise and predict
X_custom = vectorizer.transform(test_sentences)
predictions = classifier.predict(X_custom)

for sentence, prediction in zip(test_sentences, predictions):
    print(f"'{sentence}'\n  → {prediction}\n")

'I need a cheap hotel in the centre'
  → find_hotel

'Can you book it for 2 nights for 3 people'
  → book_hotel

'I am looking for a guesthouse with free parking'
  → find_hotel

'Please make a reservation for monday'
  → book_hotel



In [10]:
import pickle
import os

os.makedirs("../data/models", exist_ok=True)

# Save the vectoriser and classifier
with open("../data/models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("../data/models/intent_classifier.pkl", "wb") as f:
    pickle.dump(classifier, f)

print("Models saved to data/models/")

Models saved to data/models/


In [ ]:
# nlu.py
# Natural Language Understanding module.
# Takes raw user input and returns the intent and extracted slots.

import pickle
import os

class IntentClassifier:
    def __init__(self, model_path="../data/models/intent_classifier.pkl",
                 vectorizer_path="../data/models/tfidf_vectorizer.pkl"):
        # Load the saved classifier and vectoriser from disk
        with open(model_path, "rb") as f:
            self.classifier = pickle.load(f)
        with open(vectorizer_path, "rb") as f:
            self.vectorizer = pickle.load(f)

    def predict(self, text):
        # Convert text to TF-IDF vector and predict intent
        X = self.vectorizer.transform([text])
        intent = self.classifier.predict(X)[0]
        confidence = self.classifier.predict_proba(X).max()
        return {
            "intent": intent,
            "confidence": round(float(confidence), 2)
        }


if __name__ == "__main__":
    # Quick test when running this file directly
    clf = IntentClassifier()
    test = "I need a cheap hotel in the north"
    print(clf.predict(test))

{'intent': 'find_hotel', 'confidence': np.float64(0.92)}
